# 입낚 YOLOv8 학습 노트북

물고기 · 입낚볼 · 입낚키링을 감지하는 YOLOv8n 모델을 학습하고 ONNX 로 내보냅니다.

| class id | 이름 | 설명 |
|---|---|---|
| 0 | `fish` | 물고기 |
| 1 | `ipnak-ball` | 입낚볼 (지름 40mm 구) |
| 2 | `ipnak-keyring` | 입낚키링 (지름 40mm 원판) |

**⚠️ 클래스 순서는 절대 바꾸지 마세요.** `src/lib/yolo/types.ts` 의 `YOLO_CLASSES` 와 일치해야 합니다.

---

## 사용 전 준비

1. 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 설정하세요. (무료)
2. 데이터셋을 준비합니다. 아래 둘 중 하나를 쓰면 됩니다.
   - **옵션 A**: 입낚 관리자 → AI 학습 관리 → 라벨링 → `YOLO 학습셋 내보내기` 로 받은 zip
   - **옵션 B**: Roboflow 프로젝트에서 YOLOv8 형식으로 내보내기
3. 셀을 위에서부터 순서대로 실행합니다.

## 1. 환경 확인 및 설치

In [ ]:
!nvidia-smi
%pip install -q ultralytics onnx onnxruntime

import ultralytics
ultralytics.checks()

## 2-A. 데이터셋 준비 — 입낚 관리자 zip 사용

왼쪽 파일 탭에 zip 을 업로드한 뒤 아래 셀을 실행하거나, 실행 후 나오는 업로드 창에서 파일을 선택하세요.

zip 안의 구조는 다음과 같습니다.

```
data.yaml
images/train/*.jpg   labels/train/*.txt
images/val/*.jpg     labels/val/*.txt
```

In [ ]:
import os, glob, zipfile, shutil

DATASET_DIR = "/content/ipnak-dataset"

# 이미 업로드된 zip 을 찾고, 없으면 업로드 창을 띄운다
zips = sorted(glob.glob("/content/*.zip"))
if not zips:
    from google.colab import files
    uploaded = files.upload()
    zips = [f"/content/{name}" for name in uploaded.keys() if name.endswith(".zip")]

assert zips, "zip 파일을 찾지 못했습니다. 관리자 화면에서 학습셋을 내보내 업로드하세요."

src_zip = zips[-1]
print("사용할 zip:", src_zip)

shutil.rmtree(DATASET_DIR, ignore_errors=True)
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(src_zip) as zf:
    zf.extractall(DATASET_DIR)

DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

# data.yaml 의 path 를 절대경로로 바꿔 준다 (Ultralytics 가 상대경로를 헷갈리지 않도록)
with open(DATA_YAML, "r", encoding="utf-8") as f:
    text = f.read()
text = text.replace("path: .", f"path: {DATASET_DIR}")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(text)

print(text)
print("train 이미지:", len(glob.glob(f"{DATASET_DIR}/images/train/*")))
print("val   이미지:", len(glob.glob(f"{DATASET_DIR}/images/val/*")))

## 2-B. 데이터셋 준비 — Roboflow 사용 (옵션 A 대신)

Roboflow 에서 라벨링했다면 이 셀을 대신 실행하세요.
`Versions → Export Dataset → YOLOv8 → show download code` 에서 나오는 값을 채워 넣습니다.

**클래스 순서가 `fish, ipnak-ball, ipnak-keyring` 인지 반드시 확인하세요.**
다르면 내려받은 `data.yaml` 의 `names` 를 직접 수정해야 합니다.

In [ ]:
# 필요할 때만 실행 (2-A 를 썼다면 건너뛰세요)
#
# %pip install -q roboflow
# from roboflow import Roboflow
#
# rf = Roboflow(api_key="여기에_API_KEY")
# project = rf.workspace("워크스페이스").project("프로젝트ID")
# dataset = project.version(1).download("yolov8")
# DATA_YAML = f"{dataset.location}/data.yaml"
# print(DATA_YAML)

## 3. 학습

- `yolov8n` (nano)은 브라우저 추론에 적합한 가장 가벼운 모델입니다. 정확도가 부족하면 `yolov8s` 를 시도하세요.
- `imgsz=640` 은 **반드시 유지**하세요. `src/lib/yolo` 의 전처리가 640 기준입니다.
- 데이터가 적으면 `epochs` 를 늘리고, 과적합이 보이면 `patience` 로 조기 종료합니다.

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
IMG_SIZE = 640   # 변경 금지 — 앱 전처리와 맞춰야 한다
BATCH = 16

model = YOLO("yolov8n.pt")   # COCO 사전학습 가중치에서 시작

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=25,          # 25 에폭 동안 개선 없으면 조기 종료
    project="/content/runs",
    name="ipnak",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
)

## 4. 성능 확인

`mAP50` 이 0.8 이상이면 실사용에 무리가 없는 편입니다.
클래스별 지표를 보고 특정 클래스가 낮으면 그 클래스의 학습 이미지를 더 모으세요.

In [ ]:
metrics = model.val()
print("mAP50    :", round(float(metrics.box.map50), 4))
print("mAP50-95 :", round(float(metrics.box.map), 4))

for i, name in model.names.items():
    print(f"  [{i}] {name:16s} mAP50={float(metrics.box.maps[i]):.4f}")

In [ ]:
# 학습 곡선 / 혼동행렬 확인
from IPython.display import Image, display
import os

RUN_DIR = "/content/runs/ipnak"
for f in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p):
        print(f)
        display(Image(filename=p, width=760))

## 5. ONNX 내보내기

브라우저(onnxruntime-web)에서 쓰기 위한 설정입니다.

- `opset=12` — onnxruntime-web 호환성이 가장 안정적입니다.
- `simplify=True` — 그래프를 정리해 추론이 빨라집니다.
- `dynamic=False` — 입력 크기를 640×640 으로 고정합니다 (앱 전처리와 일치).
- **NMS 는 넣지 않습니다.** 앱의 `postprocess.ts` 가 직접 NMS 를 수행합니다.

In [ ]:
onnx_path = model.export(
    format="onnx",
    opset=12,
    simplify=True,
    dynamic=False,
    imgsz=IMG_SIZE,
    nms=False,
)
print("내보낸 파일:", onnx_path)

In [ ]:
# 출력 텐서 형태 확인 — [1, 4+nc, N] 또는 [1, N, 4+nc] 여야 앱에서 파싱된다
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print("입력:", inp.name, inp.shape)
print("출력:", out.name, out.shape)

dummy = np.zeros((1, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
res = sess.run(None, {inp.name: dummy})[0]
print("실제 출력 shape:", res.shape, "— 4+3=7 이 포함되어야 합니다")

## 6. 모델 내려받기 → 입낚에 배포

아래 셀을 실행하면 `best.onnx` 가 다운로드됩니다.

**배포 방법**: 입낚 관리자 → **AI 학습 관리 → 모델 관리 → 새 모델 업로드** 에서 이 파일을 올리면
`public/models/best.onnx` 가 교체되고 AI 카메라에 즉시 적용됩니다.

In [ ]:
import shutil
from google.colab import files

shutil.copy(str(onnx_path), "/content/best.onnx")
files.download("/content/best.onnx")

---

## 참고 — 정확도가 낮을 때

| 증상 | 대처 |
|---|---|
| 물고기는 잡는데 입낚볼을 못 잡음 | 입낚볼이 나온 사진을 더 모아 라벨링 (클래스 불균형) |
| 전반적으로 낮음 | 학습 이미지 확대 (클래스당 최소 300~500장 권장), `epochs` 증가 |
| 학습은 좋은데 실제가 나쁨 | 실제 촬영 환경(물가·역광·젖은 손)의 사진을 추가. 유저 피드 사진 선별 탭 활용 |
| 추론이 느림 | `yolov8n` 유지, `imgsz` 는 640 고정. 그래도 느리면 기기 성능 문제 |

학습 이미지가 늘어날 때마다 재학습 → 재업로드하면 서비스 정확도가 계속 올라갑니다.